In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('comments.csv')

In [3]:
# Inspect class distribution
print("\nDataset Class Distribution:")
print(df['cyberbullying_type'].value_counts())



Dataset Class Distribution:
cyberbullying_type
not_cyberbullying      775
ethnicity              209
gender                 205
other_cyberbullying    205
religion               204
age                    203
Name: count, dtype: int64


In [4]:
# 2. Define features (X) and target labels (y)
X = df['tweet_text'].astype(str)
y = df['cyberbullying_type']

In [5]:
# 3. Train-Test Split (Stratified to maintain class proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"\nTraining set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")


Training set size: 1440 samples
Testing set size: 361 samples


In [6]:
# 4. Build a Machine Learning Pipeline
# TF-IDF Vectorizer + Logistic Regression with class balancing
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=15000, 
        ngram_range=(1, 2), 
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        class_weight='balanced', 
        max_iter=1000, 
        C=2.0, 
        random_state=42
    ))
])

print("\n2. Training the model...")
pipeline.fit(X_train, y_train)

print("\n3. Evaluating model performance on test set...")
y_pred = pipeline.predict(X_test)

# Print detailed metrics report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


2. Training the model...

3. Evaluating model performance on test set...

Classification Report:
                     precision    recall  f1-score   support

                age       0.97      0.90      0.94        41
          ethnicity       0.78      0.86      0.82        42
             gender       0.89      0.80      0.85        41
  not_cyberbullying       0.96      0.99      0.98       155
other_cyberbullying       0.85      0.83      0.84        41
           religion       0.95      0.93      0.94        41

           accuracy                           0.92       361
          macro avg       0.90      0.89      0.89       361
       weighted avg       0.92      0.92      0.92       361



In [7]:




# 4. Optional: Quick text prediction test
print("\n4. Running quick prediction tests on custom strings:")
test_samples = [
    "boomers need to step down from office already",
    "can someone help me configure a docker container?",
    "go back to your country nobody wants u here",
    "your outfit looks like a trash bag",
    "women shouldn't work in engineering fields"
]

predictions = pipeline.predict(test_samples)
for text, pred in zip(test_samples, predictions):
    print(f"Text: '{text}' --> Predicted Class: [{pred}]")


4. Running quick prediction tests on custom strings:
Text: 'boomers need to step down from office already' --> Predicted Class: [age]
Text: 'can someone help me configure a docker container?' --> Predicted Class: [not_cyberbullying]
Text: 'go back to your country nobody wants u here' --> Predicted Class: [ethnicity]
Text: 'your outfit looks like a trash bag' --> Predicted Class: [other_cyberbullying]
Text: 'women shouldn't work in engineering fields' --> Predicted Class: [gender]
